# AED v3 final — corrigido e otimizado para Colab/Raspberry Pi

Correções principais: parser ASVP, reconstrução completa de `meta`, split por grupos antes do augmentation, entrada TFLite única, MobileNetV3Small, limiares conservadores e verificação de paridade Keras × TFLite.

A meta é atingir aproximadamente 90% de acurácia sem aceitar vazamento de dados ou exportação inconsistente.


In [ ]:
!ls -lh /content/aed_v3_final/processado

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install kagglehub librosa==0.11.0 soundfile scikit-learn pandas matplotlib seaborn tqdm pyyaml webrtcvad-wheels

In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any
from collections import Counter
from google.colab import files
from IPython.display import Audio, display
from tqdm.auto import tqdm

import json, random, shutil, subprocess, zipfile, os, re, time
import kagglehub
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
import tensorflow as tf
import webrtcvad

from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score, accuracy_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass

## 1. Configuração

In [ ]:
@dataclass(frozen=True)
class ConfiguracaoAED:
    sr: int = 16000
    duracao: float = 2.0
    passo_treino: float = 1.0
    passo_inferencia: float = 0.5
    n_fft: int = 512
    hop_length: int = 256
    n_mels: int = 96
    n_mfcc: int = 40
    imagem: int = 128

    max_fontes_por_classe: int = 700
    max_janelas_por_fonte: int = 2
    copias_aug_treino: int = 1

    epocas_cabeca: int = 12
    epocas_fino: int = 22
    lote: int = 32
    paciencia: int = 7
    seed: int = 42

    usar_urban8k: bool = True
    usar_scream_dataset: bool = True

    minimo_consecutivas_alerta: int = 2
    max_falsos_alertas_por_hora_alvo: float = 30.0

    meta_acuracia: float = 0.90
    meta_f1_macro: float = 0.85


CFG = ConfiguracaoAED()

CLASSES = [
    "ambiente_normal",
    "grito_panico",
    "choro_distress",
    "voz_agressiva",
    "impacto",
    "vidro_quebrando",
    "tiro",
]

ID = {classe: indice for indice, classe in enumerate(CLASSES)}
PERIGO = {classe: int(classe != "ambiente_normal") for classe in CLASSES}

RAIZ = Path("/content/aed_v3_final")
DADOS = RAIZ / "datasets"
PROC = RAIZ / "processado"
MODELOS = RAIZ / "modelos"
RESULT = RAIZ / "resultados"
ENTREGA = RAIZ / "entrega"

for pasta in (RAIZ, DADOS, PROC, MODELOS, RESULT, ENTREGA):
    pasta.mkdir(parents=True, exist_ok=True)

random.seed(CFG.seed)
np.random.seed(CFG.seed)
tf.random.set_seed(CFG.seed)

display(pd.DataFrame([asdict(CFG)]))


## 2. Baixar datasets

In [ ]:
# Kaggle
ASVP = Path(kagglehub.dataset_download('dejolilandry/asvpesdspeech-nonspeech-emotional-utterances'))
GUNSHOT = Path(kagglehub.dataset_download('emrahaydemr/gunshot-audio-dataset'))
print('ASVP:', ASVP)
print('Gunshot:', GUNSHOT)
if CFG.usar_urban8k:
    URBAN = Path(kagglehub.dataset_download('chrisfilo/urbansound8k'))
    print('UrbanSound8K:', URBAN)
else:
    URBAN = None
if CFG.usar_scream_dataset:
    try:
        SCREAM = Path(kagglehub.dataset_download('aananehsansiam/audio-dataset-of-scream-and-non-scream'))
        print('Scream/non-scream:', SCREAM)
    except Exception as e:
        SCREAM = None
        print('Falha no scream dataset, seguindo sem ele:', e)
else:
    SCREAM = None

In [ ]:
# ESC-50 e TUT Rare
URL_ESC = 'https://github.com/karolpiczak/ESC-50/archive/refs/heads/master.zip'
URL_TUT = 'https://zenodo.org/records/401395/files/TUT-rare-sound-events-2017-development.source_data_events.zip?download=1'
ESC_DIR = DADOS/'esc50'
TUT_DIR = DADOS/'tut_rare'

def baixar(url, dst):
    if dst.exists() and dst.stat().st_size>0: return
    subprocess.run(['wget','-c',url,'-O',str(dst)], check=True)

def extrair(zip_path, dst):
    if (dst/'.extraido').exists(): return
    with zipfile.ZipFile(zip_path) as z: z.extractall(dst)
    (dst/'.extraido').touch()

baixar(URL_ESC, DADOS/'esc50.zip'); extrair(DADOS/'esc50.zip', ESC_DIR)
baixar(URL_TUT, DADOS/'tut_eventos.zip'); extrair(DADOS/'tut_eventos.zip', TUT_DIR)
print('ESC/TUT prontos')

## 3. Construir metadados unificados

In [ ]:
EXT_AUDIO = {".wav", ".mp3", ".ogg", ".flac", ".m4a"}


def audios_em(pasta: Path | None) -> list[Path]:
    if pasta is None or not pasta.exists():
        return []
    return sorted(
        caminho
        for caminho in pasta.rglob("*")
        if caminho.is_file() and caminho.suffix.lower() in EXT_AUDIO
    )


def adicionar(
    registros: list[dict[str, Any]],
    arquivo: Path,
    classe: str,
    fonte: str,
    grupo: str,
    origem: str = "",
    extra: str = "",
) -> None:
    arquivo = Path(arquivo)
    if arquivo.exists() and classe in ID:
        registros.append(
            {
                "arquivo": str(arquivo.resolve()),
                "classe": classe,
                "fonte": fonte,
                "grupo": str(grupo),
                "origem_id": str(origem),
                "extra": str(extra),
            }
        )


registros = []

# ASVP-ESD: emoção = terceiro identificador numérico.
MAPA_ASVP = {
    "02": "ambiente_normal",
    "04": "choro_distress",
    "05": "voz_agressiva",
    "06": "grito_panico",
}

contagem_asvp = Counter()

for arquivo in audios_em(ASVP):
    numeros = re.findall(r"\d+", arquivo.stem)
    codigo_emocao = None
    codigo_ator = arquivo.parent.name

    if len(numeros) >= 3:
        candidato = numeros[2].zfill(2)
        if candidato in MAPA_ASVP:
            codigo_emocao = candidato

    if len(numeros) >= 6:
        codigo_ator = numeros[5].zfill(3)

    if codigo_emocao is None:
        texto = str(arquivo).lower()
        termos_por_codigo = {
            "02": ("neutral", "calm", "boredom"),
            "04": ("sad", "cry", "sobb"),
            "05": ("anger", "angry", "growl", "frustr"),
            "06": ("fear", "scream", "panic"),
        }
        for codigo, termos in termos_por_codigo.items():
            if any(termo in texto for termo in termos):
                codigo_emocao = codigo
                break

    if codigo_emocao not in MAPA_ASVP:
        continue

    classe = MAPA_ASVP[codigo_emocao]
    adicionar(
        registros,
        arquivo,
        classe,
        "ASVP",
        f"asvp:ator_{codigo_ator}",
        arquivo.stem,
        codigo_emocao,
    )
    contagem_asvp[classe] += 1

print("ASVP processado:", dict(contagem_asvp))
if contagem_asvp["voz_agressiva"] == 0:
    raise RuntimeError("O ASVP não gerou dados de voz_agressiva.")

# Gunshot Kaggle
for arquivo in audios_em(GUNSHOT):
    adicionar(
        registros,
        arquivo,
        "tiro",
        "GunshotKaggle",
        f"gunshot:{arquivo.parent.name}:{arquivo.stem}",
        arquivo.stem,
    )

# ESC-50
csvs_esc = list(ESC_DIR.rglob("esc50.csv"))
pastas_audio_esc = [p for p in ESC_DIR.rglob("audio") if p.is_dir()]

if not csvs_esc or not pastas_audio_esc:
    raise RuntimeError("ESC-50 incompleto.")

esc = pd.read_csv(csvs_esc[0])
esc_audio = pastas_audio_esc[0]

MAPA_ESC = {
    "crying_baby": "choro_distress",
    "glass_breaking": "vidro_quebrando",
    "door_wood_knock": "impacto",
    "door_wood_creaks": "impacto",
    "clapping": "impacto",
    "hand_saw": "impacto",
}

NORMAIS_ESC = {
    "rain", "sea_waves", "crickets", "chirping_birds", "water_drops",
    "wind", "footsteps", "laughing", "breathing", "snoring",
    "clock_tick", "washing_machine", "vacuum_cleaner", "engine", "train",
    "airplane", "helicopter", "thunderstorm", "siren", "keyboard_typing",
    "mouse_click", "can_opening", "brushing_teeth", "fireworks", "chainsaw",
}

for linha in esc.itertuples():
    categoria = str(linha.category)
    if categoria in MAPA_ESC:
        classe = MAPA_ESC[categoria]
    elif categoria in NORMAIS_ESC:
        classe = "ambiente_normal"
    else:
        continue

    adicionar(
        registros,
        esc_audio / str(linha.filename),
        classe,
        "ESC-50",
        f"esc50:{linha.src_file}",
        linha.filename,
        categoria,
    )

# TUT Rare
for arquivo in audios_em(TUT_DIR):
    texto = str(arquivo).lower().replace("-", "_").replace(" ", "_")
    if any(t in texto for t in ("babycry", "baby_cry", "babycrying", "baby_crying")):
        classe = "choro_distress"
    elif any(t in texto for t in ("glassbreak", "glass_break", "glassbreaking", "glass_breaking")):
        classe = "vidro_quebrando"
    elif any(t in texto for t in ("gunshot", "gun_shot", "gunfire")):
        classe = "tiro"
    else:
        continue
    adicionar(
        registros,
        arquivo,
        classe,
        "TUT_Rare",
        f"tut:{arquivo.stem}",
        arquivo.stem,
    )

# UrbanSound8K
if URBAN is not None:
    csvs_urban = list(URBAN.rglob("UrbanSound8K.csv"))
    MAPA_URBAN = {
        "gun_shot": "tiro",
        "jackhammer": "impacto",
        "drilling": "impacto",
        "air_conditioner": "ambiente_normal",
        "children_playing": "ambiente_normal",
        "street_music": "ambiente_normal",
        "engine_idling": "ambiente_normal",
        "car_horn": "ambiente_normal",
        "siren": "ambiente_normal",
        "dog_bark": "ambiente_normal",
    }

    if csvs_urban:
        csv_urban = csvs_urban[0]
        tabela_urban = pd.read_csv(csv_urban)
        raiz_audio = csv_urban.parent.parent / "audio"

        for _, linha in tabela_urban.iterrows():
            nome_classe = str(linha["class"])
            classe = MAPA_URBAN.get(nome_classe)
            if classe is None:
                continue
            arquivo = raiz_audio / f"fold{int(linha['fold'])}" / str(linha["slice_file_name"])
            adicionar(
                registros,
                arquivo,
                classe,
                "UrbanSound8K",
                f"urban:fold{int(linha['fold'])}:{linha['slice_file_name']}",
                str(linha["slice_file_name"]),
                nome_classe,
            )

# Scream / non-scream
if SCREAM is not None:
    for arquivo in audios_em(SCREAM):
        texto = str(arquivo).lower().replace("-", "_").replace(" ", "_")
        if any(t in texto for t in ("non_scream", "nonscream", "no_scream", "not_scream")):
            classe = "ambiente_normal"
        elif "scream" in texto:
            classe = "grito_panico"
        else:
            continue
        adicionar(
            registros,
            arquivo,
            classe,
            "ScreamDataset",
            f"scream:{arquivo.parent.name}:{arquivo.stem}",
            arquivo.stem,
        )

meta = pd.DataFrame(registros)
if meta.empty:
    raise RuntimeError("Nenhum áudio foi encontrado.")

meta = meta.drop_duplicates(subset=["arquivo", "classe"])

partes = []
for classe in CLASSES:
    parte = meta[meta["classe"] == classe].copy()
    grupos = parte["grupo"].drop_duplicates().tolist()
    random.Random(CFG.seed + ID[classe]).shuffle(grupos)
    grupos = grupos[: CFG.max_fontes_por_classe]
    partes.append(parte[parte["grupo"].isin(grupos)])

meta = pd.concat(partes, ignore_index=True)

resumo_meta = (
    meta.groupby("classe")
    .agg(arquivos=("arquivo", "count"), grupos=("grupo", "nunique"))
    .reindex(CLASSES, fill_value=0)
)
display(resumo_meta)

faltantes = [
    classe
    for classe in CLASSES
    if resumo_meta.loc[classe, "arquivos"] == 0
]
if faltantes:
    raise RuntimeError(f"Classes sem dados: {faltantes}")

meta.to_csv(RESULT / "fontes_unificadas.csv", index=False)


## 4. Auditoria rápida por áudio

In [ ]:
for classe in CLASSES:
    print("\n" + "=" * 72)
    print("Classe:", classe)

    parte = meta[meta["classe"] == classe]
    amostras = parte.sample(
        min(2, len(parte)),
        random_state=CFG.seed + ID[classe],
    )

    for linha in amostras.itertuples():
        print(linha.fonte, Path(linha.arquivo).name, linha.extra)
        display(Audio(linha.arquivo))


In [ ]:
APROVAR_AUDITORIA = True
if not APROVAR_AUDITORIA:
    raise RuntimeError('Ouça as amostras. Depois altere APROVAR_AUDITORIA para True e continue.')

## 5. Split por grupo antes do augmentation

In [ ]:
def dividir_grupos(grupos: list[str], semente: int):
    grupos = list(dict.fromkeys(grupos))
    random.Random(semente).shuffle(grupos)

    total = len(grupos)
    if total < 3:
        raise RuntimeError("Cada classe precisa de ao menos 3 grupos.")

    n_treino = max(1, int(round(total * 0.70)))
    n_validacao = max(1, int(round(total * 0.15)))

    if n_treino + n_validacao >= total:
        n_treino = total - 2
        n_validacao = 1

    corte = n_treino + n_validacao
    return (
        set(grupos[:n_treino]),
        set(grupos[n_treino:corte]),
        set(grupos[corte:]),
    )

# --- START OF MODIFICATION ---

# Get all unique groups from the entire meta DataFrame
all_unique_groups = meta["grupo"].drop_duplicates().tolist()

# Use dividir_grupos to split ALL unique groups globally
# This ensures that each 'grupo' is assigned to only one split across all classes.
global_train_groups, global_val_groups, global_test_groups = dividir_grupos(
    all_unique_groups,
    CFG.seed # Use a consistent seed for global splitting
)

# Create a mapping from group to its assigned split
group_to_split = {}
for g in global_train_groups:
    group_to_split[g] = "treino"
for g in global_val_groups:
    group_to_split[g] = "validacao"
for g in global_test_groups:
    group_to_split[g] = "teste"

# Apply the global split assignment to the entire meta DataFrame
meta_split = meta.copy()
meta_split["split"] = meta_split["grupo"].map(group_to_split)

# --- END OF MODIFICATION ---

# The rest of the original code for validation remains the same
# Replaced the original loop that built 'linhas' and then concatenated
# with the direct assignment using map.

tabela_split = (
    meta_split.groupby(["classe", "split"])["grupo"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(CLASSES, fill_value=0)
)
display(tabela_split)

for classe in CLASSES:
    for split in ("treino", "validacao", "teste"):
        if tabela_split.loc[classe, split] == 0:
            raise RuntimeError(f"{classe} ficou sem dados em {split}.")

vazamento = meta_split.groupby("grupo")["split"].nunique().gt(1).sum()
print("Grupos com vazamento:", int(vazamento))
if vazamento:
    raise RuntimeError("Vazamento entre splits.")

meta_split.to_csv(RESULT / "fontes_com_split.csv", index=False)

## 6. Extração de features: imagem única 128×128×3

In [ ]:
AMOSTRAS=int(CFG.sr*CFG.duracao)

def carregar_audio(path):
    y,_=librosa.load(str(path), sr=CFG.sr, mono=True)
    y=np.nan_to_num(y.astype(np.float32))
    if np.max(np.abs(y))>1: y=y/np.max(np.abs(y))
    return y

def ajustar(y):
    if len(y)<AMOSTRAS: return np.pad(y,(0,AMOSTRAS-len(y)))
    return y[:AMOSTRAS]

def selecionar_janelas(y, classe):
    if len(y)<=AMOSTRAS: return [ajustar(y)]
    passo=int(CFG.sr*CFG.passo_treino)
    j=[y[i:i+AMOSTRAS] for i in range(0,len(y)-AMOSTRAS+1,passo)]
    if len(j)<=CFG.max_janelas_por_fonte: return [ajustar(x) for x in j]
    if classe=='ambiente_normal':
        idx=np.linspace(0,len(j)-1,CFG.max_janelas_por_fonte,dtype=int)
    else:
        en=[np.sqrt(np.mean(x*x)+1e-9) for x in j]
        idx=np.argsort(en)[-CFG.max_janelas_por_fonte:]
    return [ajustar(j[int(i)]) for i in sorted(idx)]

def aug(y, rng):
    z=y.copy()
    z*=10**(rng.uniform(-9,6)/20)
    z=np.roll(z, int(rng.uniform(-0.15,0.15)*len(z)))
    if rng.random()<0.65:
        snr=rng.uniform(8,28)
        ps=np.mean(z*z)+1e-10
        pn=ps/(10**(snr/10))
        z+=rng.normal(0,np.sqrt(pn),size=z.shape)
    if rng.random()<0.25:
        rate=rng.uniform(0.92,1.08)
        try:
            zz=librosa.effects.time_stretch(z, rate=rate)
            z=ajustar(zz)
        except Exception: pass
    m=np.max(np.abs(z))
    if m>1: z=z/m
    return z.astype(np.float32)

def norm_img(x):
    x=x.astype(np.float32)
    lo,hi=np.percentile(x,1),np.percentile(x,99)
    x=np.clip((x-lo)/(hi-lo+1e-6),0,1)
    return x

def resize_mat(x, size=128):
    return cv2.resize(x, (size,size), interpolation=cv2.INTER_AREA)

import cv2

def feature_image(y):
    y=ajustar(y)
    mel=librosa.feature.melspectrogram(y=y, sr=CFG.sr, n_fft=CFG.n_fft, hop_length=CFG.hop_length, n_mels=CFG.n_mels, power=2.0)
    logmel=librosa.power_to_db(mel, ref=np.max, top_db=80)
    mfcc=librosa.feature.mfcc(S=logmel, n_mfcc=CFG.n_mfcc)
    delta=librosa.feature.delta(logmel)
    c1=resize_mat(norm_img(logmel), CFG.imagem)
    c2=resize_mat(norm_img(mfcc), CFG.imagem)
    c3=resize_mat(norm_img(delta), CFG.imagem)
    return np.stack([c1,c2,c3], axis=-1).astype(np.float16)

def processar(split):
    cache=PROC/f'{split}.npz'; meta_cache=PROC/f'{split}.csv'
    if cache.exists() and meta_cache.exists():
        z=np.load(cache); return z['X'], z['y'], pd.read_csv(meta_cache)
    parte=meta_split[meta_split.split==split]
    rng=np.random.default_rng(CFG.seed+{'treino':1,'validacao':2,'teste':3}[split])
    X=[]; y=[]; regs=[]
    for r in tqdm(parte.itertuples(), total=len(parte), desc=split):
        try:
            audio=carregar_audio(r.arquivo)
            janelas=selecionar_janelas(audio, r.classe)
        except Exception:
            continue
        for ji,j in enumerate(janelas):
            versoes=[('original',j)]
            if split=='treino':
                for k in range(CFG.copias_aug_treino): versoes.append((f'aug{k+1}', aug(j,rng)))
            for v,a in versoes:
                X.append(feature_image(a)); y.append(ID[r.classe]); regs.append({'split':split,'classe':r.classe,'grupo':r.grupo,'arquivo':r.arquivo,'janela':ji,'versao':v})
    X=np.asarray(X,dtype=np.float16); y=np.asarray(y,dtype=np.int64); regs=pd.DataFrame(regs)
    np.savez_compressed(cache, X=X, y=y); regs.to_csv(meta_cache,index=False)
    return X,y,regs

X_train,y_train,meta_train=processar('treino')
X_val,y_val,meta_val=processar('validacao')
X_test,y_test,meta_test=processar('teste')
print(X_train.shape, X_val.shape, X_test.shape)


## 7. Modelo MobileNetV2

In [ ]:
base = tf.keras.applications.MobileNetV3Small(
    input_shape=(CFG.imagem, CFG.imagem, 3),
    include_top=False,
    weights="imagenet",
    include_preprocessing=True,
)
base.trainable = False

entrada = tf.keras.Input(
    shape=(CFG.imagem, CFG.imagem, 3),
    name="audio_features",
)

x = entrada * 255.0
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.40)(x)
x = tf.keras.layers.Dense(160, activation="relu")(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.30)(x)
saida = tf.keras.layers.Dense(
    len(CLASSES),
    activation="softmax",
    name="evento",
)(x)

model = tf.keras.Model(entrada, saida, name="aed_v3_mobilenetv3")
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name="top2"),
    ],
)
model.summary()


## 8. Treino da cabeça e ajuste fino

In [ ]:
# Substitua toda a célula de treinamento por esta versão.
# Ela NÃO mantém X_train/X_val inteiros na RAM durante o model.fit.
# Os arquivos .npz são extraídos diretamente para .npy e lidos por memmap.

import gc
import math
import shutil
import zipfile

import numpy as np
import tensorflow as tf

from sklearn.metrics import confusion_matrix
from sklearn.utils.class_weight import compute_class_weight


# ============================================================
# 1. Transformar os caches NPZ em NPY sem carregar tudo na RAM
# ============================================================

def extrair_array_do_npz(
    arquivo_npz,
    nome_interno: str,
    destino,
) -> None:
    destino = Path(destino)

    if destino.exists() and destino.stat().st_size > 0:
        print("Já existe:", destino)
        return

    temporario = destino.with_suffix(
        destino.suffix + ".part"
    )

    temporario.unlink(missing_ok=True)

    with zipfile.ZipFile(
        arquivo_npz,
        "r",
    ) as compactado:
        nomes = compactado.namelist()

        if nome_interno not in nomes:
            raise KeyError(
                f"{nome_interno} não existe em {arquivo_npz}. "
                f"Conteúdo: {nomes}"
            )

        with compactado.open(
            nome_interno,
            "r",
        ) as origem:
            with temporario.open(
                "wb",
            ) as saida:
                shutil.copyfileobj(
                    origem,
                    saida,
                    length=8 * 1024 * 1024,
                )

    temporario.replace(
        destino
    )

    print(
        "Extraído:",
        destino,
        f"{destino.stat().st_size / 1024**3:.3f} GB",
    )


def preparar_memmap(
    split: str,
):
    arquivo_npz = (
        PROC / f"{split}.npz"
    )

    if not arquivo_npz.exists():
        raise FileNotFoundError(
            f"Cache ausente: {arquivo_npz}. "
            "Execute primeiro a célula de extração das features."
        )

    caminho_x = (
        PROC / f"{split}_X.npy"
    )
    caminho_y = (
        PROC / f"{split}_y.npy"
    )

    extrair_array_do_npz(
        arquivo_npz,
        "X.npy",
        caminho_x,
    )
    extrair_array_do_npz(
        arquivo_npz,
        "y.npy",
        caminho_y,
    )

    X = np.load(
        caminho_x,
        mmap_mode="r",
    )
    y = np.load(
        caminho_y,
        mmap_mode="r",
    )

    return X, y


# Libera os arrays gigantes criados pela célula anterior.
for nome_variavel in (
    "X_train",
    "X_val",
    "X_test",
    "y_train",
    "y_val",
    "y_test",
):
    if nome_variavel in globals():
        del globals()[
            nome_variavel
        ]

gc.collect()

X_train_mm, y_train_mm = preparar_memmap(
    "treino"
)
X_val_mm, y_val_mm = preparar_memmap(
    "validacao"
)
X_test_mm, y_test_mm = preparar_memmap(
    "teste"
)

print("\nMemmaps criados:")

for nome, array in (
    ("X_train", X_train_mm),
    ("X_val", X_val_mm),
    ("X_test", X_test_mm),
):
    print(
        f"{nome:8s}: "
        f"{array.shape} "
        f"{array.dtype} "
        "(arquivo em disco)"
    )


# ============================================================
# 2. Dataset em lotes lidos do disco
# ============================================================

pesos = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(
        len(CLASSES)
    ),
    y=np.asarray(
        y_train_mm
    ),
)

class_weight = np.asarray(
    pesos,
    dtype=np.float32,
)

LOTE_CABECA = min(
    int(CFG.lote),
    8,
)
LOTE_AJUSTE = min(
    int(CFG.lote),
    4,
)
LOTE_VALIDACAO = 8


def criar_dataset_memmap(
    X_mm,
    y_mm,
    tamanho_lote: int,
    embaralhar: bool,
    usar_pesos: bool,
    semente: int,
):
    quantidade = len(
        y_mm
    )

    def gerador():
        indices = np.arange(
            quantidade,
            dtype=np.int64,
        )

        if embaralhar:
            rng = np.random.default_rng(
                semente
            )
            rng.shuffle(
                indices
            )

        for inicio in range(
            0,
            quantidade,
            tamanho_lote,
        ):
            indices_lote = indices[
                inicio:
                inicio + tamanho_lote
            ]

            # Apenas este lote é copiado para a RAM.
            imagens = np.asarray(
                X_mm[indices_lote],
                dtype=np.float16,
            )
            rotulos = np.asarray(
                y_mm[indices_lote],
                dtype=np.int32,
            )

            if usar_pesos:
                pesos_lote = class_weight[
                    rotulos
                ]

                yield (
                    imagens,
                    rotulos,
                    pesos_lote,
                )
            else:
                yield (
                    imagens,
                    rotulos,
                )

    if usar_pesos:
        assinatura = (
            tf.TensorSpec(
                shape=(
                    None,
                    CFG.imagem,
                    CFG.imagem,
                    3,
                ),
                dtype=tf.float16,
            ),
            tf.TensorSpec(
                shape=(None,),
                dtype=tf.int32,
            ),
            tf.TensorSpec(
                shape=(None,),
                dtype=tf.float32,
            ),
        )
    else:
        assinatura = (
            tf.TensorSpec(
                shape=(
                    None,
                    CFG.imagem,
                    CFG.imagem,
                    3,
                ),
                dtype=tf.float16,
            ),
            tf.TensorSpec(
                shape=(None,),
                dtype=tf.int32,
            ),
        )

    dataset = tf.data.Dataset.from_generator(
        gerador,
        output_signature=assinatura,
    )

    opcoes = tf.data.Options()
    opcoes.experimental_deterministic = (
        not embaralhar
    )

    dataset = dataset.with_options(
        opcoes
    )

    # Não use AUTOTUNE aqui.
    # Prefetch de apenas um lote evita novos picos de RAM.
    dataset = dataset.prefetch(
        1
    )

    passos = math.ceil(
        quantidade / tamanho_lote
    )

    return dataset, passos


ds_treino_cabeca, passos_cabeca = (
    criar_dataset_memmap(
        X_train_mm,
        y_train_mm,
        LOTE_CABECA,
        embaralhar=True,
        usar_pesos=True,
        semente=CFG.seed,
    )
)

ds_treino_ajuste, passos_ajuste = (
    criar_dataset_memmap(
        X_train_mm,
        y_train_mm,
        LOTE_AJUSTE,
        embaralhar=True,
        usar_pesos=True,
        semente=CFG.seed + 1,
    )
)

ds_validacao, passos_validacao = (
    criar_dataset_memmap(
        X_val_mm,
        y_val_mm,
        LOTE_VALIDACAO,
        embaralhar=False,
        usar_pesos=False,
        semente=CFG.seed,
    )
)

print(
    "\nLotes:",
    {
        "cabeca": LOTE_CABECA,
        "ajuste": LOTE_AJUSTE,
        "validacao": LOTE_VALIDACAO,
    },
)


# ============================================================
# 3. Métrica macro-F1 sem executar outro predict
# ============================================================

@tf.keras.utils.register_keras_serializable(
    package="ViolenceEdgeAI"
)
class SparseMacroF1(
    tf.keras.metrics.Metric
):
    def __init__(
        self,
        num_classes: int,
        name: str = "macro_f1",
        **kwargs,
    ):
        super().__init__(
            name=name,
            **kwargs,
        )

        self.num_classes = int(
            num_classes
        )

        self.matriz = self.add_weight(
            name="matriz_confusao",
            shape=(
                self.num_classes,
                self.num_classes,
            ),
            initializer="zeros",
            dtype=tf.float32,
        )

    def update_state(
        self,
        y_true,
        y_pred,
        sample_weight=None,
    ):
        y_true = tf.cast(
            tf.reshape(
                y_true,
                [-1],
            ),
            tf.int32,
        )

        y_pred = tf.argmax(
            y_pred,
            axis=-1,
            output_type=tf.int32,
        )

        matriz_lote = (
            tf.math.confusion_matrix(
                y_true,
                y_pred,
                num_classes=(
                    self.num_classes
                ),
                dtype=tf.float32,
            )
        )

        self.matriz.assign_add(
            matriz_lote
        )

    def result(self):
        vp = tf.linalg.diag_part(
            self.matriz
        )

        fp = (
            tf.reduce_sum(
                self.matriz,
                axis=0,
            )
            - vp
        )

        fn = (
            tf.reduce_sum(
                self.matriz,
                axis=1,
            )
            - vp
        )

        precisao = (
            tf.math.divide_no_nan(
                vp,
                vp + fp,
            )
        )

        revocacao = (
            tf.math.divide_no_nan(
                vp,
                vp + fn,
            )
        )

        f1 = tf.math.divide_no_nan(
            2.0
            * precisao
            * revocacao,
            precisao + revocacao,
        )

        presentes = tf.cast(
            tf.reduce_sum(
                self.matriz,
                axis=1,
            ) > 0,
            tf.float32,
        )

        return tf.math.divide_no_nan(
            tf.reduce_sum(
                f1 * presentes
            ),
            tf.reduce_sum(
                presentes
            ),
        )

    def reset_state(self):
        self.matriz.assign(
            tf.zeros_like(
                self.matriz
            )
        )

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "num_classes": (
                    self.num_classes
                )
            }
        )
        return config


def compilar_modelo(
    modelo,
    taxa_aprendizado: float,
):
    modelo.compile(
        optimizer=(
            tf.keras.optimizers.Adam(
                taxa_aprendizado
            )
        ),
        loss=(
            "sparse_categorical_crossentropy"
        ),
        metrics=[
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            ),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(
                k=2,
                name="top2",
            ),
            SparseMacroF1(
                len(CLASSES)
            ),
        ],
        jit_compile=False,
        run_eagerly=False,
    )


# ============================================================
# 4. Recriar o modelo após limpar grafos antigos
# ============================================================

if "model" in globals():
    del model

if "base" in globals():
    del base

gc.collect()
tf.keras.backend.clear_session()


base = (
    tf.keras.applications.MobileNetV3Small(
        input_shape=(
            CFG.imagem,
            CFG.imagem,
            3,
        ),
        include_top=False,
        weights="imagenet",
        include_preprocessing=True,
    )
)

base.trainable = False

entrada = tf.keras.Input(
    shape=(
        CFG.imagem,
        CFG.imagem,
        3,
    ),
    dtype=tf.float16,
    name="audio_features",
)

# As features estão em [0, 1].
# O MobileNetV3 com include_preprocessing=True espera [0, 255].
x = tf.keras.layers.Lambda(
    lambda inputs: tf.cast(inputs, tf.float32) * 255.0,
    name="cast_and_scale_input"
)(entrada)

x = base(
    x,
    training=False,
)
x = (
    tf.keras.layers.GlobalAveragePooling2D()
    (x)
)
x = tf.keras.layers.Dropout(
    0.40
)(x)
x = tf.keras.layers.Dense(
    160,
    activation="relu",
)(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(
    0.30
)(x)

saida = tf.keras.layers.Dense(
    len(CLASSES),
    activation="softmax",
    dtype=tf.float32,
    name="evento",
)(x)

model = tf.keras.Model(
    entrada,
    saida,
    name="aed_v3_mobilenetv3",
)


# ============================================================
# 5. Treinar a cabeça
# ============================================================

PESO_CABECA = (
    MODELOS
    / "aed_v3_cabeca.weights.h5"
)

PESO_AJUSTE = (
    MODELOS
    / "aed_v3_ajuste.weights.h5"
)

MODELO_FINAL = (
    MODELOS
    / "aed_v3_melhor.keras"
)

compilar_modelo(
    model,
    1e-3,
)

callbacks_cabeca = [
    tf.keras.callbacks.ModelCheckpoint(
        PESO_CABECA,
        monitor="val_macro_f1",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_macro_f1",
        mode="max",
        patience=CFG.paciencia,
        restore_best_weights=False,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(
        RESULT
        / "historico_cabeca.csv"
    ),
]

hist1 = model.fit(
    ds_treino_cabeca,
    validation_data=ds_validacao,
    epochs=CFG.epocas_cabeca,
    steps_per_epoch=passos_cabeca,
    validation_steps=passos_validacao,
    callbacks=callbacks_cabeca,
    verbose=1,
)

if not PESO_CABECA.exists():
    raise FileNotFoundError(
        "Checkpoint da cabeça não foi criado."
    )

model.load_weights(
    PESO_CABECA
)


# ============================================================
# 6. Ajuste fino parcial
# ============================================================

base.trainable = True

for camada in base.layers:
    camada.trainable = False

for camada in base.layers[-24:]:
    if not isinstance(
        camada,
        tf.keras.layers.BatchNormalization,
    ):
        camada.trainable = True

compilar_modelo(
    model,
    5e-5,
)

callbacks_ajuste = [
    tf.keras.callbacks.ModelCheckpoint(
        PESO_AJUSTE,
        monitor="val_macro_f1",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_macro_f1",
        mode="max",
        patience=CFG.paciencia,
        restore_best_weights=False,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(
        RESULT
        / "historico_ajuste_fino.csv"
    ),
]

hist2 = model.fit(
    ds_treino_ajuste,
    validation_data=ds_validacao,
    epochs=CFG.epocas_fino,
    steps_per_epoch=passos_ajuste,
    validation_steps=passos_validacao,
    callbacks=callbacks_ajuste,
    verbose=1,
)

if not PESO_AJUSTE.exists():
    raise FileNotFoundError(
        "Checkpoint do ajuste não foi criado."
    )


# ============================================================
# 7. Escolher cabeça ou ajuste fino pela validação
# ============================================================

def avaliar_macro_f1_dataset(
    modelo,
    dataset,
    passos: int,
) -> float:
    matriz = np.zeros(
        (
            len(CLASSES),
            len(CLASSES),
        ),
        dtype=np.int64,
    )

    iterador = iter(
        dataset
    )

    for _ in range(passos):
        imagens, rotulos = next(
            iterador
        )

        probabilidades = modelo(
            imagens,
            training=False,
        ).numpy()

        previstos = np.argmax(
            probabilidades,
            axis=1,
        )

        matriz += confusion_matrix(
            rotulos.numpy(),
            previstos,
            labels=np.arange(
                len(CLASSES)
            ),
        )

    vp = np.diag(
        matriz
    ).astype(np.float64)
    fp = matriz.sum(
        axis=0
    ) - vp
    fn = matriz.sum(
        axis=1
    ) - vp

    precisao = np.divide(
        vp,
        vp + fp,
        out=np.zeros_like(vp),
        where=(vp + fp) > 0,
    )
    revocacao = np.divide(
        vp,
        vp + fn,
        out=np.zeros_like(vp),
        where=(vp + fn) > 0,
    )
    f1 = np.divide(
        2 * precisao * revocacao,
        precisao + revocacao,
        out=np.zeros_like(vp),
        where=(
            precisao + revocacao
        ) > 0,
    )

    presentes = (
        matriz.sum(axis=1) > 0
    )

    return float(
        f1[presentes].mean()
    )


model.load_weights(
    PESO_CABECA
)
f1_cabeca = avaliar_macro_f1_dataset(
    model,
    ds_validacao,
    passos_validacao,
)

model.load_weights(
    PESO_AJUSTE
)
f1_ajuste = avaliar_macro_f1_dataset(
    model,
    ds_validacao,
    passos_validacao,
)

print(
    "Macro-F1 cabeça:",
    f"{f1_cabeca:.4f}",
)
print(
    "Macro-F1 ajuste:",
    f"{f1_ajuste:.4f}",
)

if f1_cabeca > f1_ajuste:
    model.load_weights(
        PESO_CABECA
    )
    etapa_selecionada = "cabeca"
    melhor_f1 = f1_cabeca
else:
    model.load_weights(
        PESO_AJUSTE
    )
    etapa_selecionada = (
        "ajuste_fino"
    )
    melhor_f1 = f1_ajuste

model.save(
    MODELO_FINAL
)

print(
    "Etapa selecionada:",
    etapa_selecionada,
)
print(
    "Macro-F1 selecionado:",
    f"{melhor_f1:.4f}",
)
print(
    "Modelo salvo em:",
    MODELO_FINAL,
)


# ============================================================
# 8. Deixar o teste disponível para a célula seguinte
# ============================================================

X_test = X_test_mm
y_test = y_test_mm

gc.collect()

print(
    "Teste pronto por memmap:",
    X_test.shape,
    y_test.shape,
)


## 9. Avaliação por janela e por gravação

In [ ]:
probs=model.predict(X_test,batch_size=CFG.lote,verbose=1)
pred=probs.argmax(axis=1)
print(classification_report(y_test,pred,target_names=CLASSES,digits=4))
metrics_janela={'accuracy':accuracy_score(y_test,pred), 'balanced_accuracy':balanced_accuracy_score(y_test,pred), 'f1_macro':f1_score(y_test,pred,average='macro'), 'f1_weighted':f1_score(y_test,pred,average='weighted')}
print(json.dumps(metrics_janela, indent=2))
pd.DataFrame(classification_report(y_test,pred,target_names=CLASSES,output_dict=True)).T.to_csv(RESULT/'metricas_janela.csv')
cm=confusion_matrix(y_test,pred)
plt.figure(figsize=(10,8)); sns.heatmap(cm,annot=True,fmt='d',xticklabels=CLASSES,yticklabels=CLASSES,cmap='Blues'); plt.xticks(rotation=45,ha='right'); plt.title('Matriz de confusão - janela'); plt.tight_layout(); plt.savefig(RESULT/'matriz_confusao_janela.png',dpi=160); plt.show()

# agregação por gravação: média das probabilidades por grupo
pred_df=meta_test.copy(); pred_df['real_id']=y_test
for i,c in enumerate(CLASSES): pred_df[f'p_{c}']=probs[:,i]
linhas=[]
for grupo, g in pred_df.groupby('grupo'):
    p=np.array([g[f'p_{c}'].mean() for c in CLASSES])
    real=int(g['real_id'].mode().iloc[0]); pr=int(p.argmax())
    linhas.append({'grupo':grupo,'real':real,'previsto':pr,'real_nome':CLASSES[real],'previsto_nome':CLASSES[pr],'confianca':float(p.max())})
grav=pd.DataFrame(linhas)
metrics_grav={'accuracy':accuracy_score(grav.real,grav.previsto), 'balanced_accuracy':balanced_accuracy_score(grav.real,grav.previsto), 'f1_macro':f1_score(grav.real,grav.previsto,average='macro'), 'f1_weighted':f1_score(grav.real,grav.previsto,average='weighted')}
print('Por gravação:', json.dumps(metrics_grav, indent=2))
grav.to_csv(RESULT/'predicoes_por_gravacao.csv', index=False)
pd.DataFrame([metrics_janela, metrics_grav], index=['janela','gravacao']).to_csv(RESULT/'resumo_metricas.csv')

## 10. Limiares por classe e decisão binária perigo

In [ ]:
probs_val = model.predict(ds_validacao, steps=passos_validacao, verbose=0)

limiares = {}
registros = []

for indice, classe in enumerate(CLASSES):
    real = (y_val_mm == indice).astype(int)
    melhor_limiar = 0.50
    melhor_f1 = -1.0

    for limiar in np.linspace(0.30, 0.95, 66):
        previsto = (probs_val[:, indice] >= limiar).astype(int)
        valor_f1 = f1_score(real, previsto, zero_division=0)

        if valor_f1 > melhor_f1:
            melhor_f1 = float(valor_f1)
            melhor_limiar = float(limiar)

    if classe == "ambiente_normal":
        melhor_limiar = 0.50
    else:
        melhor_limiar = max(0.58, melhor_limiar)

    limiares[classe] = round(melhor_limiar, 3)
    registros.append(
        {
            "classe": classe,
            "limiar": limiares[classe],
            "f1_validacao": melhor_f1,
        }
    )

pd.DataFrame(registros).to_csv(RESULT / "limiares.csv", index=False)
display(pd.DataFrame(registros))

real_bin = np.array(
    [0 if CLASSES[i] == "ambiente_normal" else 1 for i in y_test]
)
pred_bin = np.array(
    [0 if CLASSES[i] == "ambiente_normal" else 1 for i in pred]
)

print("Binário perigo vs normal")
print(
    classification_report(
        real_bin,
        pred_bin,
        target_names=["normal", "perigo"],
        digits=4,
    )
)


## 11. Auditoria de falsos alertas em ambiente normal

In [ ]:
# normal só do teste
normais=meta_test[meta_test.classe=='ambiente_normal'].copy()
if not normais.empty:
    idx=normais.index.to_numpy()
    probs_norm=probs[idx]
    pred_norm=pred[idx]
    ativos=[]
    for p,pr in zip(probs_norm,pred_norm):
        classe=CLASSES[int(pr)]
        ativos.append(classe!='ambiente_normal' and p[int(pr)]>=limiares[classe])
    falsos=sum(ativos)
    horas=len(idx)*CFG.duracao/3600
    taxa=falsos/horas if horas>0 else 0
    auditoria={'janelas_normais':int(len(idx)), 'horas_estimadas':float(horas), 'falsos_alertas_janela':int(falsos), 'falsos_alertas_por_hora':float(taxa)}
    print(json.dumps(auditoria,indent=2))
    (RESULT/'auditoria_falsos_alertas.json').write_text(json.dumps(auditoria,indent=2), encoding='utf-8')
    print('Alvo falsos alertas/h <=', CFG.max_falsos_alertas_por_hora_alvo, 'OK:', taxa <= CFG.max_falsos_alertas_por_hora_alvo)

## 12. Exportação TFLite com uma entrada única

In [ ]:
model.save(MODELOS/'aed_v3_final.keras')
conv=tf.lite.TFLiteConverter.from_keras_model(model)
tflite=conv.convert(); (MODELOS/'aed_v3_float32.tflite').write_bytes(tflite)
conv=tf.lite.TFLiteConverter.from_keras_model(model); conv.optimizations=[tf.lite.Optimize.DEFAULT]; conv.target_spec.supported_types=[tf.float16]
tflite16=conv.convert(); (MODELOS/'aed_v3_float16.tflite').write_bytes(tflite16)

def rep_data():
    for i in range(min(300,len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]
try:
    conv=tf.lite.TFLiteConverter.from_keras_model(model); conv.optimizations=[tf.lite.Optimize.DEFAULT]; conv.representative_dataset=rep_data
    conv.target_spec.supported_ops=[tf.lite.OpsSet.TFLITE_BUILTINS_INT8]; conv.inference_input_type=tf.int8; conv.inference_output_type=tf.int8
    int8=conv.convert(); (MODELOS/'aed_v3_int8.tflite').write_bytes(int8); print('INT8 exportado')
except Exception as e:
    print('INT8 falhou:', e)
for f in MODELOS.glob('*'):
    if f.is_file(): print(f.name, f.stat().st_size/1e6, 'MB')

## 13. Paridade Keras × TFLite

In [ ]:
def predict_tflite(path: Path, X: np.ndarray) -> np.ndarray:
    interpretador = tf.lite.Interpreter(model_path=str(path))
    interpretador.allocate_tensors()

    entrada = interpretador.get_input_details()[0]
    saida = interpretador.get_output_details()[0]
    predicoes = []

    for imagem in X:
        valor = imagem[np.newaxis, ...].astype(np.float32)

        if entrada["dtype"] in (np.int8, np.uint8):
            escala, zero = entrada["quantization"]
            if escala == 0:
                raise RuntimeError("Entrada quantizada sem escala.")
            valor = np.round(valor / escala + zero)
            limites = np.iinfo(entrada["dtype"])
            valor = np.clip(valor, limites.min, limites.max).astype(
                entrada["dtype"]
            )
        else:
            valor = valor.astype(entrada["dtype"])

        interpretador.set_tensor(entrada["index"], valor)
        interpretador.invoke()
        predicao = interpretador.get_tensor(saida["index"])

        if saida["dtype"] in (np.int8, np.uint8):
            escala, zero = saida["quantization"]
            predicao = (predicao.astype(np.float32) - zero) * escala

        predicoes.append(predicao[0])

    return np.asarray(predicoes)


amostra = X_test[: min(250, len(X_test))]
pred_keras = model.predict(amostra, verbose=0)
relatorio = []

for nome in (
    "aed_v3_float32.tflite",
    "aed_v3_float16.tflite",
    "aed_v3_int8.tflite",
):
    caminho = MODELOS / nome
    if not caminho.exists():
        continue

    pred_tflite = predict_tflite(caminho, amostra)
    concordancia = float(
        np.mean(
            np.argmax(pred_tflite, axis=1)
            == np.argmax(pred_keras, axis=1)
        )
    )
    diferenca = float(np.max(np.abs(pred_tflite - pred_keras)))

    relatorio.append(
        {
            "modelo": nome,
            "concordancia_argmax": concordancia,
            "diferenca_maxima": diferenca,
        }
    )

paridade_df = pd.DataFrame(relatorio)
display(paridade_df)
paridade_df.to_csv(RESULT / "paridade_tflite.csv", index=False)

float_models = paridade_df[
    paridade_df["modelo"].isin(
        ["aed_v3_float32.tflite", "aed_v3_float16.tflite"]
    )
]

if not float_models.empty and float_models["concordancia_argmax"].min() < 0.99:
    raise RuntimeError("Float32/Float16 ficaram abaixo de 99% de paridade.")


## 14. Teste em áudio externo e VAD paralelo

In [ ]:
enviados=files.upload()
auds=[n for n in enviados if Path(n).suffix.lower() in EXT_AUDIO]
if len(auds)!=1: raise RuntimeError('Envie exatamente um áudio.')
TESTE=Path(auds[0]).resolve()
display(Audio(str(TESTE)))

def janelas_deslizantes(y, passo_seg):
    passo=int(CFG.sr*passo_seg)
    if len(y)<=AMOSTRAS: return [(0,ajustar(y))]
    return [(i/CFG.sr, y[i:i+AMOSTRAS]) for i in range(0,len(y)-AMOSTRAS+1,passo)]

def inferir(path):
    y=carregar_audio(path); js=janelas_deslizantes(y, CFG.passo_inferencia)
    X=np.asarray([feature_image(j) for _,j in js], dtype=np.float32)
    pr=model.predict(X,verbose=0)
    linhas=[]
    for (t,_),p in zip(js,pr):
        cid=int(p.argmax()); cls=CLASSES[cid]; conf=float(p[cid]); ativo=cls!='ambiente_normal' and conf>=limiares[cls]
        linhas.append({'inicio':t,'fim':t+CFG.duracao,'classe':cls,'confianca':conf,'limiar':limiares[cls],'ativo':ativo})
    df=pd.DataFrame(linhas)
    cons=0; last=None; alertas=[]
    for r in df.itertuples():
        if r.ativo and r.classe==last: cons+=1
        elif r.ativo: cons=1
        else: cons=0
        last=r.classe if r.ativo else None
        alertas.append(cons>=CFG.minimo_consecutivas_alerta)
    df['alerta_temporal']=alertas
    return df

tl=inferir(TESTE); display(tl.sort_values('confianca',ascending=False).head(20)); tl.to_csv(RESULT/'timeline_audio_externo.csv',index=False)

# VAD não bloqueia AED; só decide se Whisper/NLP deve rodar
def vad_info(path):
    y=carregar_audio(path); pcm=(np.clip(y,-1,1)*32767).astype(np.int16)
    vad=webrtcvad.Vad(2); frame_ms=30; n=int(CFG.sr*frame_ms/1000); dec=[]
    for i in range(0,len(pcm)-n+1,n): dec.append(vad.is_speech(pcm[i:i+n].tobytes(), CFG.sr))
    return {'frames':len(dec),'frames_fala':int(sum(dec)),'proporcao_fala':float(np.mean(dec)) if dec else 0}
print(vad_info(TESTE))

## 15. Pacote final

In [ ]:
if ENTREGA.exists(): shutil.rmtree(ENTREGA)
ENTREGA.mkdir(parents=True)
for f in MODELOS.glob('*'):
    if f.is_file(): shutil.copy2(f, ENTREGA/f.name)
for f in RESULT.glob('*'):
    if f.is_file(): shutil.copy2(f, ENTREGA/f.name)
config={'configuracao':asdict(CFG),'classes':CLASSES,'limiares':limiares,'metricas_janela':metrics_janela,'metricas_gravacao':metrics_grav}
(ENTREGA/'configuracao_aed_v3.json').write_text(json.dumps(config,ensure_ascii=False,indent=2), encoding='utf-8')
zip_final=shutil.make_archive('/content/aed_v3_entrega','zip',root_dir=str(ENTREGA))
print('ZIP:', zip_final)
files.download(zip_final)

## Critério de conclusão

O modelo só é candidato à Raspberry Pi quando:

- todas as classes possuem dados;
- não há vazamento entre grupos;
- métricas são calculadas no teste separado;
- Float32 e Float16 apresentam pelo menos 99% de paridade;
- os falsos alertas são medidos;
- áudios externos, de preferência gravados pelos INMP441, também funcionam.

Acurácia de 90% é uma meta experimental, não uma garantia.
